# Ablation decision and mechanism diagnostics

Use this notebook for one completed ablation root—the directory containing `per_seed.csv` and the aggregate `summary.json`. The figures mirror the repository's decision rules:

1. show paired quality changes for every seed;
2. test efficiency only alongside quality non-inferiority;
3. inspect interventions, recurrent-pass refinement, and teacher-forced schedule gap for a plausible mechanism;
4. keep the statistical claim modest with three seeds: medians and paired outcomes, not decorative confidence bands.

Diagnostics support interpretation, but do not replace the predeclared quality and efficiency criteria.

In [ ]:
from collections import defaultdict
from pathlib import Path
from statistics import median
import sys

import matplotlib
if "ipykernel" in sys.modules:
    matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
REPO_ROOT = HERE if (HERE / "experiments").exists() else HERE.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from figures.plotting_utils import (
    discover_ablation_roots,
    grouped,
    load_ablation_rows,
    load_diagnostic_records,
    metric_label,
    paired_values,
    read_json,
    set_plot_style,
)

set_plot_style()
RESULT_ROOT = REPO_ROOT / "results" / "ablations"
FIGURE_DIR = REPO_ROOT / "figures"


In [ ]:
candidates = discover_ablation_roots(RESULT_ROOT) if RESULT_ROOT.exists() else []
for index, path in enumerate(candidates):
    print(index, path.relative_to(REPO_ROOT))

# Select explicitly after inspecting the list. The default is merely convenient.
ABLATION_ROOT = candidates[-1] if candidates else RESULT_ROOT / "memory_gate_init" / "RUN_ID"
print("selected:", ABLATION_ROOT)

In [ ]:
rows = load_ablation_rows(ABLATION_ROOT)
summary_path = ABLATION_ROOT / "summary.json"
summary = read_json(summary_path) if summary_path.exists() else {}
CONTROL = summary.get("control", "control")
VARIANTS = list(summary.get("variants", {}))
TREATMENT = VARIANTS[0] if VARIANTS else "treatment"
QUALITY_METRIC = (
    summary.get("variants", {}).get(TREATMENT, {}).get("quality_metric")
    or "drift.append_recurrent.token_legality"
)
print(f"rows={len(rows)} control={CONTROL!r} treatment={TREATMENT!r}")
print("quality metric:", QUALITY_METRIC)
if TREATMENT in summary.get("variants", {}):
    print(summary["variants"][TREATMENT])

## Paired quality is the primary decision plot

Lines connect the same data seed. The right panel shows treatment-minus-control deltas and the ±1 percentage-point quality margin. A mean can be dominated by one seed; the repository decision uses the median and counts how many paired seeds agree.

In [ ]:
pairs = paired_values(rows, control=CONTROL, treatment=TREATMENT, metric=QUALITY_METRIC)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
for seed, control_value, treatment_value in pairs:
    axes[0].plot([0, 1], [control_value, treatment_value], "o-", alpha=0.7, label=f"seed {seed}")
axes[0].set_xticks([0, 1], [CONTROL, TREATMENT])
axes[0].set_ylabel(metric_label(QUALITY_METRIC))
axes[0].set_title("Paired final quality")
axes[0].legend(fontsize=8)

deltas = [treatment - control for _seed, control, treatment in pairs]
axes[1].axhspan(-0.01, 0.01, color="grey", alpha=0.12, label="±1 pp margin")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].scatter(range(len(deltas)), deltas, s=65)
if deltas:
    axes[1].hlines(median(deltas), -0.35, len(deltas) - 0.65, color="tab:red", linewidth=2.5,
                   label=f"median {median(deltas):+.3f}")
axes[1].set_xticks(range(len(pairs)), [f"seed {seed}" for seed, *_ in pairs])
axes[1].set_ylabel("Treatment − control")
axes[1].set_title("Paired quality delta")
axes[1].legend(fontsize=8)
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{ABLATION_ROOT.name}_paired_quality.png", dpi=220, bbox_inches="tight")

## Quality–efficiency trade-off

The x-axis is the per-seed treatment/control ratio for append-recurrent generated-token throughput. The dashed vertical line marks a 10% throughput improvement; the horizontal line marks the −1 percentage-point non-inferiority boundary. Points in the upper-right region satisfy both conditions for that seed. Parameter and memory-byte reductions are reported separately below because they are not timing measurements.

In [ ]:
throughput_metric = "drift.append_recurrent.eval_output_tok_per_s"
quality_lookup = {(str(row.get("variant")), str(row.get("seed"))): row.get(QUALITY_METRIC) for row in rows}
throughput_lookup = {(str(row.get("variant")), str(row.get("seed"))): row.get(throughput_metric) for row in rows}

points = []
for seed, control_quality, treatment_quality in pairs:
    control_speed = throughput_lookup.get((CONTROL, seed))
    treatment_speed = throughput_lookup.get((TREATMENT, seed))
    if isinstance(control_speed, (int, float)) and isinstance(treatment_speed, (int, float)) and control_speed > 0:
        points.append((seed, treatment_speed / control_speed, treatment_quality - control_quality))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
axes[0].axvline(1.10, color="tab:green", linestyle="--", label="10% speed win")
axes[0].axhline(-0.01, color="tab:red", linestyle="--", label="quality margin")
for seed, ratio, delta in points:
    axes[0].scatter(ratio, delta, s=70)
    axes[0].annotate(f"seed {seed}", (ratio, delta), xytext=(5, 5), textcoords="offset points", fontsize=8)
axes[0].set(xlabel="Append throughput ratio (treatment / control)",
            ylabel="Paired quality delta", title="Measured quality–speed trade-off")
axes[0].legend(fontsize=8)

size_metrics = [
    ("model.non_embedding_parameters", "Non-embedding parameters"),
    ("model_config.memory_bytes_per_token", "memory bytes / token"),
]
x = 0
labels = []
for metric, label in size_metrics:
    metric_pairs = paired_values(rows, control=CONTROL, treatment=TREATMENT, metric=metric)
    ratios = [right / left for _seed, left, right in metric_pairs if left > 0]
    if ratios:
        axes[1].scatter([x] * len(ratios), ratios, alpha=0.65)
        axes[1].hlines(median(ratios), x - 0.25, x + 0.25, color="black", linewidth=2)
    labels.append(label)
    x += 1
axes[1].axhline(0.75, color="tab:green", linestyle="--", label="25% reduction")
axes[1].axhline(1.0, color="black", linewidth=1)
axes[1].set_xticks(range(len(labels)), labels, rotation=15)
axes[1].set_ylabel("Treatment / control ratio")
axes[1].set_title("Structural efficiency (not runtime)")
axes[1].legend(fontsize=8)
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{ABLATION_ROOT.name}_quality_efficiency.png", dpi=220, bbox_inches="tight")

## Memory interventions

Bars show NLL increase relative to the correct memory. Zeroing, cross-example substitution, causal resampling, prefix averaging, and extra lag test different kinds of memory dependence. A near-zero delta suggests that intervention did not remove information the model currently uses—it does not prove the memory is useless in all contexts.

In [ ]:
diagnostic_rows = load_diagnostic_records(ABLATION_ROOT)
for row in diagnostic_rows:
    try:
        row["variant"] = Path(row["run_dir"]).resolve().relative_to(ABLATION_ROOT.resolve()).parts[0]
    except (ValueError, IndexError):
        row["variant"] = "unknown"
print(f"Loaded {len(diagnostic_rows)} diagnostic summaries")

In [ ]:
interventions = [
    "zero_memory_bank",
    "cross_example",
    "causal_position_resample",
    "causal_prefix_mean",
    "extra_lag",
]
fig, ax = plt.subplots(figsize=(10.5, 4.8))
variants = [CONTROL, TREATMENT]
width = 0.36
for variant_index, variant in enumerate(variants):
    medians = []
    for name in interventions:
        key = f"memory_interventions.loss_deltas.{name}"
        values = [row.get(key) for row in diagnostic_rows
                  if row.get("variant") == variant and isinstance(row.get(key), (int, float))]
        medians.append(median(values) if values else float("nan"))
    positions = [index + (variant_index - 0.5) * width for index in range(len(interventions))]
    ax.bar(positions, medians, width=width, label=variant, alpha=0.8)
ax.axhline(0, color="black", linewidth=1)
ax.set_xticks(range(len(interventions)), [name.replace("_", "\n") for name in interventions])
ax.set_ylabel("NLL increase from correct memory")
ax.set_title("Median causal-memory intervention effect")
ax.legend()
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{ABLATION_ROOT.name}_memory_interventions.png", dpi=220, bbox_inches="tight")

## Pass dynamics

Trained passes are left of the vertical divider; untrained extra passes are right of it. Continued improvement suggests iterative refinement generalizes beyond the training depth. Oscillation or degradation suggests the learned update is tied to a particular finite pass schedule.

In [ ]:
DIAGNOSTIC_VARIANT = TREATMENT
DIAGNOSTIC_SEED = "1337"
chosen = next((row for row in diagnostic_rows
               if row.get("variant") == DIAGNOSTIC_VARIANT and str(row.get("seed")) == DIAGNOSTIC_SEED), None)
print("diagnostic run:", None if chosen is None else chosen["run_dir"])

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.1), sharex=True)
if chosen is None:
    for ax in axes:
        ax.text(0.5, 0.5, "No selected diagnostic run", ha="center", va="center")
        ax.set_axis_off()
else:
    dynamics = chosen["payload"]["pass_dynamics"]
    trained = dynamics["trained_passes"]
    extra = dynamics["extra_passes"]
    all_passes = [*trained, *extra]
    split = len(trained) + 0.5
    fields = [
        ("loss", "NLL"),
        ("logit_kl_from_previous", "Logit KL from previous pass"),
        ("hidden_rms_delta", "Hidden-state RMS update"),
        ("memory.effective_rank", "Memory effective rank"),
    ]
    for ax, (field, label) in zip(axes, fields):
        def nested_value(item):
            value = item
            for part in field.split("."):
                if not isinstance(value, dict) or part not in value:
                    return None
                value = value[part]
            return value
        points = [(item["pass"], nested_value(item)) for item in all_passes if nested_value(item) is not None]
        if points:
            ax.plot(*zip(*points), marker="o")
        ax.axvline(split, color="black", linestyle="--", linewidth=1, label="training-depth boundary")
        ax.set(xlabel="Recurrent pass", ylabel=label)
    axes[0].legend(fontsize=8)
    fig.suptitle(f"Pass dynamics: {DIAGNOSTIC_VARIANT}, seed {DIAGNOSTIC_SEED}", y=1.03)
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{ABLATION_ROOT.name}_pass_dynamics.png", dpi=220, bbox_inches="tight")

## Does the treatment close the deployment schedule gap?

This final plot compares medians across seeds at each teacher-forced suffix position. A useful treatment should reduce append-minus-recompute NLL without sacrificing recompute quality; merely making both schedules equally poor is not a win.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharex=True)
for variant, color in ((CONTROL, "tab:gray"), (TREATMENT, "tab:blue")):
    by_position_nll = defaultdict(list)
    by_position_memory = defaultdict(list)
    for row in diagnostic_rows:
        if row.get("variant") != variant:
            continue
        for item in row["payload"]["teacher_forced_schedule_gap"]["positions"]:
            if item.get("count", 0) <= 0:
                continue
            position = item["generated_position"]
            by_position_nll[position].append(item["nll_delta"])
            by_position_memory[position].append(item["memory_rms_delta"])
    for ax, values, label in ((axes[0], by_position_nll, "Append − recompute NLL"),
                              (axes[1], by_position_memory, "Memory RMS distance")):
        positions = sorted(values)
        if positions:
            ax.plot(positions, [median(values[position]) for position in positions],
                    marker="o", color=color, label=variant)
        ax.set(xlabel="Gold suffix position", ylabel=label)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].legend()
fig.suptitle("Teacher-forced deployment-schedule gap", y=1.03)
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{ABLATION_ROOT.name}_schedule_gap_comparison.png", dpi=220, bbox_inches="tight")